In [21]:
import torch
import plotly.express as px
from utils import upscale_tensor
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [22]:
hr_scene_vel_tensor = torch.load("./torch_data/scene_vel_tensor.pt")

In [32]:
factor = 5
# use utils function
lr_scene_vel_tensor = upscale_tensor(hr_scene_vel_tensor, upscaling_factor = factor)

print("HR tensor shape:", hr_scene_vel_tensor.shape)
print("LR tensor shape:", lr_scene_vel_tensor.shape)

HR tensor shape: torch.Size([4, 50, 50])
LR tensor shape: torch.Size([4, 10, 10])


In [33]:
# Visualise value one
fig = px.imshow(hr_scene_vel_tensor[0], 
                color_continuous_scale = 'RdBu_r',
                origin = "upper", 
                title = "LR vel scene")
fig.show()

In [34]:
fig = make_subplots(rows = 1, cols = 2,
                    subplot_titles = ("low resolution (lr) velocity", "high resolution (hr) velocity"))

fig.add_trace(go.Heatmap(z = lr_scene_vel_tensor[0].squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_scene_vel_tensor[0].squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 2)

fig.update_layout(plot_bgcolor = 'rgba(0,0,0,0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()

## Bilinear interpolation 

Input:
Low resolution velocity  
Quey locations for high resolution


In [42]:
# Unsqueeze twice for mini-batch and channel dim
bilinear_scene = torch.nn.functional.interpolate(lr_scene_vel_tensor[0].unsqueeze(0).unsqueeze(0), scale_factor = 5, mode = 'bilinear', 
                                                 align_corners = True, recompute_scale_factor = None, antialias = False)

bilinear_scene.squeeze().shape

torch.Size([50, 50])

## Toy example

In [43]:
fig = make_subplots(rows = 1, cols = 2,
                    subplot_titles = ("low resolution (lr) velocity", "high resolution (hr) velocity"))

fig.add_trace(go.Heatmap(z = bilinear_scene.squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_scene_vel_tensor[0].squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 2)

fig.update_layout(plot_bgcolor = 'rgba(0,0,0,0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()

- Non-stationarities are not captured
- Lower values: low bias